# A.7 多輪對話和答案怎麼分配長度？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**把桌面擴大兩倍，模型就一定找得到最遠線索嗎？

context像一個有限大小的桌面，要給歷史、問題、檢索、答案各留空間。裁切順序要明確，不能最後才發現重要線索被擠掉。

**把直覺寫成數學：**T_history+T_prompt+T_retrieval+T_response≤max_length；支持長度不等於學會長距離任務。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
budget = {"history": 20, "question": 15, "retrieval": 40, "response_reserved": 25}
limit = 100
print(budget, sum(budget.values()), "fits", sum(budget.values()) <= limit)
assert sum(budget.values()) <= limit

**如何讀結果：**用真實tokenizer計數而非字數；引用資料被裁掉要重新檢查可回答性。

**只改一件事：**只把history加30，決定優先裁什麼並說明理由。
